In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn import preprocessing
from sklearn import metrics
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_fscore_support

from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn import tree
import graphviz

%matplotlib inline

In [2]:
# Read test and train data
train_acc = pd.read_csv("test data/raw_account_70_new.csv")
train_data = pd.read_csv("test data/raw_data_70_new.csv")
train_enq = pd.read_csv("test data/raw_enquiry_70_new.csv")

test_acc = pd.read_csv("test data/raw_account_30_new.csv")
test_data = pd.read_csv("test data/raw_data_30_new.csv")
test_enq = pd.read_csv("test data/raw_enquiry_30_new.csv")

features = list(train_data.columns)

/Users/yogeshsoniwal/anaconda/lib/python2.7/site-packages/IPython/core/interactiveshell.py:2717: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)
/Users/yogeshsoniwal/anaconda/lib/python2.7/site-packages/IPython/core/interactiveshell.py:2717: DtypeWarning: Columns (12,20,63) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [3]:
def data_imputation(data, means):
    
    data['feature_1']  = data['feature_1'].fillna('NA')
    data['feature_3']  = data['feature_3'].fillna(means['feature_3'])
    data['feature_4']  = data['feature_4'].fillna(3)
    data['feature_7']  = data['feature_7'].fillna(means['feature_7'])
    data['feature_11'] = data['feature_11'].fillna('Y')
    data['feature_12'] = data['feature_12'].fillna('PM1')
    data['feature_13'] = data['feature_13'].fillna('NA')
    data['feature_14'] = data['feature_14'].fillna(12)
    data['feature_15'] = data['feature_15'].fillna('SA03')
    data['feature_16'] = data['feature_16'].fillna('NA')
    data['feature_19'] = data['feature_19'].fillna(1)
    data['feature_23'] = data['feature_23'].fillna('N')
    data['feature_25'] = data['feature_25'].fillna(1)
    data['feature_26'] = data['feature_26'].fillna(0)
    data['feature_27'] = data['feature_27'].fillna('NA')
    data['feature_28'] = data['feature_28'].fillna('NA')
    data['feature_29'] = data['feature_29'].fillna(110059)
    data['feature_30'] = data['feature_30'].fillna(2010)
    data['feature_31'] = data['feature_31'].fillna(0)
    data['feature_32'] = data['feature_32'].fillna('NA')
    data['feature_33'] = data['feature_33'].fillna('Y')
    data['feature_34'] = data['feature_34'].fillna(1)
    data['feature_35'] = data['feature_35'].fillna(1)
    data['feature_36'] = data['feature_36'].fillna('NA')
    data['feature_37'] = data['feature_37'].fillna('NA')
    data['feature_38'] = data['feature_38'].fillna('NA')
    data['feature_39'] = data['feature_39'].fillna(0)
    data['feature_40'] = data['feature_40'].fillna(0)
    data['feature_41'] = data['feature_41'].fillna(0)
    data['feature_42'] = data['feature_42'].fillna(0)
    data['feature_43'] = data['feature_43'].fillna('NA')
    data['feature_44'] = data['feature_44'].fillna(201301)
    data['feature_46'] = data['feature_46'].fillna('NA')
    data['feature_48'] = data['feature_48'].fillna('NA')
    data['feature_50'] = data['feature_50'].fillna('Y')
    data['feature_51'] = data['feature_51'].fillna('NA')
    data['feature_52'] = data['feature_52'].fillna(0)
    data['feature_55'] = data['feature_55'].fillna(1)
    data['feature_56'] = data['feature_56'].fillna(10)
    data['feature_57'] = data['feature_57'].fillna('Y')
    data['feature_58'] = data['feature_58'].fillna('N')
    data['feature_59'] = data['feature_59'].fillna('Y')
    data['feature_60'] = data['feature_60'].fillna('N')
    data['feature_62'] = data['feature_62'].fillna('Y')
    data['feature_64'] = data['feature_64'].fillna(10)
    data['feature_65'] = data['feature_65'].fillna(157)
    data['feature_66'] = data['feature_66'].fillna(110059)
    data['feature_67'] = data['feature_67'].fillna(0)
    data['feature_68'] = data['feature_68'].fillna(1)
    data['feature_69'] = data['feature_69'].fillna(1)
    data['feature_71'] = data['feature_71'].fillna(10)
    data['feature_72'] = data['feature_72'].fillna('R')
    data['feature_76'] = data['feature_76'].fillna(0)
    data['feature_78'] = data['feature_78'].fillna(1)
    data['feature_79'] = data['feature_79'].fillna('N')
    
    return data

In [4]:
# These features are either same in all rows or are non-useful (such as user information (dob etc.))
features_to_remove = ['dt_opened', 'customer_no', 'entry_time', 'feature_2', 'feature_5', 'feature_6', 'feature_8',\
                  'feature_9', 'feature_10', 'feature_17', 'feature_18', 'feature_21', 'feature_22', 'feature_24',\
                  'feature_45', 'feature_49', 'feature_53', 'feature_54', 'feature_61', 'feature_63', 'feature_70',\
                  'feature_73', 'feature_74', 'feature_75', 'feature_77']

features_to_take = [f for f in features if f not in features_to_remove]

# Impute the test and train data
train_data = data_imputation(train_data, train_data.mean())
test_data = data_imputation(test_data, test_data.mean())

train_data = train_data[features_to_take]
test_data = test_data[features_to_take]

In [5]:
train_data_size = train_data.shape[0]
test_data_size = test_data.shape[0]

X_labels = [f for f in features_to_take if f not in ['Bad_label']]

X_train = train_data[X_labels]
X_test = test_data[X_labels]

y_train = train_data['Bad_label']
y_test  = test_data['Bad_label']

# Convert categorical variables to factors
cat_vars = ['feature_1', 'feature_11', 'feature_12', 'feature_13', 'feature_15', 'feature_16', 'feature_20',\
            'feature_23', 'feature_27', 'feature_28', 'feature_32', 'feature_33', 'feature_36', 'feature_37',\
           'feature_38', 'feature_43', 'feature_46', 'feature_47', 'feature_48', 'feature_50', 'feature_51',\
           'feature_57', 'feature_58', 'feature_59', 'feature_60', 'feature_62', 'feature_72', 'feature_79']

# For converting categorical variables to numeric form
encoder = preprocessing.LabelEncoder()

for var in cat_vars:
    c = encoder.fit_transform(pd.concat([X_train[var], X_test[var]]))
    X_train[var] = c[0:train_data_size]
    X_test[var] = c[train_data_size:]

# Fit decision tree classifier
#model = LogisticRegression()
#model = svm.SVC()
model = tree.DecisionTreeClassifier()
model.fit(X_train, y_train)

/Users/yogeshsoniwal/anaconda/lib/python2.7/site-packages/ipykernel/__main__.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/Users/yogeshsoniwal/anaconda/lib/python2.7/site-packages/ipykernel/__main__.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


DecisionTreeClassifier(class_weight=None, criterion='gini', max_depth=None,
            max_features=None, max_leaf_nodes=None,
            min_impurity_split=1e-07, min_samples_leaf=1,
            min_samples_split=2, min_weight_fraction_leaf=0.0,
            presort=False, random_state=None, splitter='best')

In [6]:
# Compute feature importances from decision tree classifier
# and take features with a score of more than 0.01
feture_importances = model.feature_importances_
feture_importances = feture_importances >= 0.01
final_features = [features_to_take[i] for i, f in enumerate(feture_importances) if f]

# Take feature with importance score of more than 0.01
X_train = X_train[final_features]
X_test = X_test[final_features]

# Train decision tree classifier with these features
model = tree.DecisionTreeClassifier()
model.fit(X_train, y_train)

DecisionTreeClassifier(class_weight=None, criterion='gini', max_depth=None,
            max_features=None, max_leaf_nodes=None,
            min_impurity_split=1e-07, min_samples_leaf=1,
            min_samples_split=2, min_weight_fraction_leaf=0.0,
            presort=False, random_state=None, splitter='best')

In [7]:
dot_data = tree.export_graphviz(model, out_file=None, 
                         feature_names=features_to_take,  
                         class_names='Bad_label',  
                         filled=True, rounded=True,  
                         special_characters=True)
graph = graphviz.Source(dot_data)  
#graph

In [8]:
# Predict on test data
y_pred = model.predict(X_test)
print('Accuracy on test set: {:.2f}'.format(model.score(X_test, y_test)))
print(metrics.classification_report(y_test, y_pred))

Accuracy on test set: 0.90
             precision    recall  f1-score   support

          0       0.96      0.94      0.95      9778
          1       0.06      0.09      0.07       462

avg / total       0.92      0.90      0.91     10240

